# 🎛️ 24-信号处理 · 02 采样定理与混叠

> 关键词：Nyquist、采样率、混叠、抗混叠滤波
> 为什么采样率必须够高？欠采样后 6kHz 会变成 2kHz——本章手算证明。

## 本节要解决的问题

| # | 问题 | 对应内容 |
| --- | --- | --- |
| 1 | Nyquist 采样定理说什么？ | §1 核心概念 |
| 2 | 为什么 fs<2fmax 会混叠？ | §2 手算 ✍️ |
| 3 | 工程上怎么防混叠？ | §3 深入原理 |
| 4 | 怎么用 numpy 验证序列一致？ | §4 代码实战 |

## 故事引入

录音棚要把 20kHz 的高频泛音录下来，ADC 采样率必须 ≥ 40kHz（CD 用 44.1kHz）。如果只给 32kHz，20kHz 的信号会被"伪装"成 12kHz 的假信号——这假信号就是**混叠**，混进后很难再分开。

混叠不是抽象概念，下图给出了一个可复现的真实例子：对同一连续信号用不同采样方案采样，再对采样序列做 DFT，可以看到频谱里出现了"折叠"的伪峰——那些本不存在的高频成分，被低采样率"伪装"成了低频。这就是 Nyquist 定理要防止的事情。

![混叠不是抽象概念，下图给出了一个可复现的真实例子：对同一连续](../figures/aliasing_uniform.jpg)

> **怎么读**：每个面板上半是采样方案、下半是对应的 DFT 频谱：采样越稀疏（低于 Nyquist 率），频谱中的真实峰与混叠伪峰越纠缠。它直观回答了"为什么 f_s 必须 ≥ 2f_max"。
>
> 📚 **图源**：Maciejewski MW et al. (2009) *Nonuniform sampling and spectral aliasing*, J Magn Reson 199(1):88–93（开放获取）图 1。


## 核心概念

**采样定理（Nyquist–Shannon）**：若信号最高频率 $f_{\max}$，采样率必须满足

$$f_s\ge2f_{\max}$$

采样后的离散序列才能无失真重建原信号。$f_s/2$ 称为 **Nyquist 频率**。

**混叠（aliasing）**：当 $f_s<2f_{\max}$，高频成分被折叠到 $[0,f_s/2]$ 内的假频率

$$f_{\text{alias}}=|f-k\,f_s|\quad(\text{取最小正值})$$

## ✍️ 手算欠采样混叠：6kHz 以 8kHz 采样

**设定**：$x(t)=\cos(2\pi\cdot6000\,t)$，采样率 $f_s=8000$ Hz。

**第 1 步：混叠频率**

$$f_{\text{alias}}=|6000-8000|=\mathbf{2000}\ \text{Hz}$$

**第 2 步：采样序列逐点对比**

离散序列 $x[n]=\cos(2\pi\cdot6000/8000\,n)=\cos(1.5\pi n)$；

而 2000Hz 以 8kHz 采样是 $\cos(2\pi\cdot2000/8000\,n)=\cos(0.5\pi n)$。

$$\cos(1.5\pi n)=\cos((2-0.5)\pi n)=\cos(2\pi n-0.5\pi n)=\cos(0.5\pi n)$$

**逐点**：n=0: 1/1；n=1: 0/0；n=2: −1/−1；n=3: 0/0；n=4: 1/1——**两个序列完全相同**！

**手算收获**：

- 混叠不是"失真"而是"伪装"：6kHz 在 8kHz 采样后和 2kHz 的采样序列**逐点不可分**——任何 DSP 都救不回来
- 余弦的混叠公式：$\cos(2\pi f/f_s\,n)$ 中 $f\to f+f_s$（或 $f\to f_s-f$）不改变序列——这就是 $f_{\text{alias}}$ 的由来
- 防混叠三板斧：**提高 fs ≥ 2fmax、先低通（抗混叠滤波）再采样、过采样+抽取**
- 面试速答：CD 44.1kHz 对应 22.05kHz 上限（人耳 20kHz 可闻）+ 余量

In [1]:
# ---- §2.1 验证：numpy 复算混叠序列 ----
import numpy as np
n = np.arange(9)
s6k = np.cos(2*np.pi*6000/8000*n)
s2k = np.cos(2*np.pi*2000/8000*n)
print('6kHz@8kHz:', np.round(s6k, 3).tolist())
print('2kHz@8kHz:', np.round(s2k, 3).tolist())
assert np.allclose(s6k, s2k)
f_alias = abs(6000 - 8000)
print('别名频率 =', f_alias, 'Hz')
assert f_alias == 2000
print('\n✓ 与手算一致')


6kHz@8kHz: [1.0, -0.0, -1.0, 0.0, 1.0, -0.0, -1.0, -0.0, 1.0]
2kHz@8kHz: [1.0, 0.0, -1.0, -0.0, 1.0, 0.0, -1.0, -0.0, 1.0]
别名频率 = 2000 Hz

✓ 与手算一致


## 深入原理：采样与重建

**采样**：$x[n]=x(nT)$，$T=1/f_s$。

**频谱关系**：采样后频谱是原频谱以 $f_s$ 为周期的搬移

$$X_s(f)=\frac{1}{T}\sum_k X(f-kf_s)$$

若 $f_s\ge2f_{\max}$，搬移后的频谱**不重叠**，用理想低通 $\text{sinc}$ 卷积即可无失真重建；若重叠（欠采样），重叠区频谱叠加 → 混叠。

**工程做法**：ADC 前放**抗混叠低通**（截止 $\approx f_s/2$），把高于 Nyquist 的成分滤掉再采样；语音 16kHz 采样即覆盖 8kHz 内带宽。

## 代码实战：画出混叠

生成两个频率的正弦，观察它们在 8kHz 采样下完全重合（可直接运行，无随机）。

In [2]:
# ---- 混叠可视化数据（不依赖 matplotlib 画窗也能跑） ----
import numpy as np
n = np.arange(0, 33)
s6k = np.cos(2*np.pi*6000/8000*n)
s2k = np.cos(2*np.pi*2000/8000*n)
diff = np.abs(s6k - s2k).max()
print('6kHz 与 2kHz 采样序列最大差 = %.2e' % diff)
print('采样序列:', np.round(s6k[:8], 2).tolist())
assert diff < 1e-12
print('✓ 两个不同频率在 8kHz 采样下完全重合（混叠）')


6kHz 与 2kHz 采样序列最大差 = 2.18e-14
采样序列: [1.0, -0.0, -1.0, 0.0, 1.0, -0.0, -1.0, -0.0]
✓ 两个不同频率在 8kHz 采样下完全重合（混叠）


## 面试考点

**Q1：采样率不够会怎样？为什么说混叠不可恢复？**

频谱搬移重叠 → 高频伪装成低频；重叠后信息丢失，理论上不可逆。

**Q2：为什么先滤波再采样而不是先采样再滤波？**

采样后的混叠分量已和真实低频重叠，滤波器无法区分；必须先低通（抗混叠）把超 Nyquist 成分滤掉。

**Q3：48kHz 能采到多高？**

Nyquist=24kHz；工程留 10% 余量。

**Q4：过采样（oversampling）有什么好处？**

把量化噪声摊到更宽频带，配合抽取可提高有效位数（噪声整形）。

## 小结与下一章预告

- Nyquist：$f_s\ge2f_{\max}$；欠采样 → 混叠伪装
- 6kHz@8kHz ≡ 2kHz@8kHz：手算逐点一致证明混叠不可分
- 下一章：采样之后怎么分析**时变信号**？——STFT 语谱图

## ✅ 自测清单

- [ ] 能说出 Nyquist 定理与 $f_{alias}$ 公式
- [ ] 能手算一个欠采样混叠频率（如 6kHz@8kHz → 2kHz）
- [ ] 知道抗混叠滤波为什么必须在 ADC 前
- [ ] 能解释频谱搬移 $X_s(f)=\frac1T\sum X(f-kf_s)$